# Dataset zawodnik-mecz dla wszystkich meczów

Ten notebook buduje główną tabelę analityczną projektu. Jeden wiersz oznacza występ jednego zawodnika w jednym meczu.

Tabela:
- cechy zawodnika per 90 minut,
- kontekst meczu,
- wynik,
- rywala,
- momentum drużyny,
- xG drużyny.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


Project root: c:\Users\misie\Documents\UAM\MGR_MK


## Importy


In [2]:
import pandas as pd

from mgr_mk.data_loader import load_world_cup_2018_matches
from mgr_mk.datasets import build_player_match_dataset


## Konfiguracja

Domyślnie brane są pod uwagę wszystkie mecze, można to zmodyfu=ikować dając FALSE.


In [3]:
RUN_ALL_MATCHES = True
NUMBER_OF_MATCHES_FOR_TEST = 10

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "tables" / "player_match_dataset.csv"


## Lista meczów


In [4]:
df_matches = load_world_cup_2018_matches()

if RUN_ALL_MATCHES:
    match_ids = df_matches["match_id"].tolist()
else:
    match_ids = df_matches.head(NUMBER_OF_MATCHES_FOR_TEST)["match_id"].tolist()

print("Liczba meczów do przeliczenia:", len(match_ids))
df_matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score", "competition_stage"]].head()


Liczba meczów do przeliczenia: 64


,match_id,match_date,home_team,away_team,home_score,away_score,competition_stage
0,8650,2018-07-06,"{'home_team_id': 781, 'home_team_name': 'Brazi...","{'away_team_id': 782, 'away_team_name': 'Belgi...",1,2,"{'id': 11, 'name': 'Quarter-finals'}"
1,7584,2018-07-02,"{'home_team_id': 782, 'home_team_name': 'Belgi...","{'away_team_id': 778, 'away_team_name': 'Japan...",3,2,"{'id': 33, 'name': 'Round of 16'}"
2,7554,2018-06-24,"{'home_team_id': 768, 'home_team_name': 'Engla...","{'away_team_id': 798, 'away_team_name': 'Panam...",6,1,"{'id': 10, 'name': 'Group Stage'}"
3,7539,2018-06-19,"{'home_team_id': 789, 'home_team_name': 'Polan...","{'away_team_id': 787, 'away_team_name': 'Seneg...",1,2,"{'id': 10, 'name': 'Group Stage'}"
4,7550,2018-06-22,"{'home_team_id': 786, 'home_team_name': 'Serbi...","{'away_team_id': 773, 'away_team_name': 'Switz...",1,2,"{'id': 10, 'name': 'Group Stage'}"


## Budowa datasetu

Ta komórka przechodzi po meczach, ładuje eventy i buduje tabelę zawodnik-mecz.


In [5]:
player_match_dataset = build_player_match_dataset(
    df_matches,
    match_ids=match_ids,
    window=5,
)

print("Rozmiar datasetu:", player_match_dataset.shape)
print("Liczba unikalnych zawodników:", player_match_dataset["player.id"].nunique())
print("Liczba drużyn:", player_match_dataset["team.name"].nunique())

player_match_dataset.head(10)


Rozmiar datasetu: (1787, 30)
Liczba unikalnych zawodników: 603
Liczba drużyn: 32


,match_id,match_date,match_week,competition_stage,player.id,player.name,team.name,opponent,venue,result,...,duels_per90,pressures_per90,interceptions_per90,total_xg_per90,pass_accuracy,events_per90,team_momentum,opponent_momentum,team_xg,opponent_xg
0,8650,2018-07-06,5,Quarter-finals,2954.0,Youri Tielemans,Belgium,Brazil,away,win,...,0.000000,90.000000,0.000000,0.000000,100.000000,154.285714,50.814505,80.251898,0.405256,2.609196
1,8650,2018-07-06,5,Quarter-finals,3054.0,Fernando Luiz Roza,Brazil,Belgium,home,loss,...,2.812500,19.687500,1.875000,0.061561,86.363636,177.187500,80.251898,50.814505,2.609196,0.405256
2,8650,2018-07-06,5,Quarter-finals,3077.0,Jan Vertonghen,Belgium,Brazil,away,win,...,3.956044,36.593407,0.989011,0.000000,85.106383,168.131868,50.814505,80.251898,0.405256,2.609196
3,8650,2018-07-06,5,Quarter-finals,3089.0,Kevin De Bruyne,Belgium,Brazil,away,win,...,0.947368,39.789474,0.000000,0.066543,78.571429,166.736842,50.814505,80.251898,0.405256,2.609196
4,8650,2018-07-06,5,Quarter-finals,3101.0,Vincent Kompany,Belgium,Brazil,away,win,...,1.000000,9.000000,0.000000,0.056956,93.181818,137.000000,50.814505,80.251898,0.405256,2.609196
5,8650,2018-07-06,5,Quarter-finals,3176.0,Thomas Meunier,Belgium,Brazil,away,win,...,3.000000,49.000000,0.000000,0.000000,75.862069,138.000000,50.814505,80.251898,0.405256,2.609196
6,8650,2018-07-06,5,Quarter-finals,3202.0,Gabriel Fernando de Jesus,Brazil,Belgium,home,loss,...,1.551724,29.482759,0.000000,0.813966,70.000000,105.517241,80.251898,50.814505,2.609196,0.405256
7,8650,2018-07-06,5,Quarter-finals,3289.0,Romelu Lukaku Menama,Belgium,Brazil,away,win,...,0.000000,14.651163,0.000000,0.000000,69.230769,94.186047,50.814505,80.251898,0.405256,2.609196
8,8650,2018-07-06,5,Quarter-finals,3295.0,Thiago Emiliano da Silva,Brazil,Belgium,home,loss,...,0.967742,2.903226,0.967742,0.265182,98.148148,162.580645,80.251898,50.814505,2.609196,0.405256
9,8650,2018-07-06,5,Quarter-finals,3296.0,Marouane Fellaini-Bakkioui,Belgium,Brazil,away,win,...,10.531915,60.319149,1.914894,0.075380,80.000000,184.787234,50.814505,80.251898,0.405256,2.609196


## Kolumny datasetu


In [6]:
pd.DataFrame({
    "column": player_match_dataset.columns,
    "non_null": [player_match_dataset[col].notna().sum() for col in player_match_dataset.columns],
    "dtype": [player_match_dataset[col].dtype for col in player_match_dataset.columns],
})


,column,non_null,dtype
0,match_id,1787,int64
1,match_date,1787,str
2,match_week,1787,int64
3,competition_stage,1787,str
4,player.id,1787,float64
5,player.name,1787,str
6,team.name,1787,str
7,opponent,1787,str
8,venue,1787,str
9,result,1787,str


## Kontrola jakości

Szybkie sprawdzenie, czy każdy mecz ma sensowne liczby zawodników oraz czy wynik i rywal są przypisane do każdego wiersza.


In [7]:
quality_by_match = (
    player_match_dataset.groupby("match_id")
    .agg(
        players=("player.id", "nunique"),
        teams=("team.name", "nunique"),
        missing_opponent=("opponent", lambda value: value.isna().sum()),
        missing_result=("result", lambda value: value.isna().sum()),
    )
    .reset_index()
)

quality_by_match.head(15)


,match_id,players,teams,missing_opponent,missing_result
0,7525,28,2,0,0
1,7529,28,2,0,0
2,7530,28,2,0,0
3,7531,28,2,0,0
4,7532,28,2,0,0
5,7533,28,2,0,0
6,7534,28,2,0,0
7,7535,28,2,0,0
8,7536,28,2,0,0
9,7537,27,2,0,0


## Podsumowania


In [8]:
player_match_dataset.groupby("team.name").agg(
    matches=("match_id", "nunique"),
    player_match_rows=("player.id", "count"),
    avg_team_xg=("team_xg", "mean"),
    avg_team_momentum=("team_momentum", "mean"),
).sort_values("avg_team_momentum", ascending=False).head(32)


,matches,player_match_rows,avg_team_xg,avg_team_momentum
team.name,,,,
Spain,4,57,2.997413,109.785716
Croatia,7,100,2.850007,97.135842
Russia,5,72,2.460196,87.909126
Germany,3,42,2.107915,85.310232
Brazil,5,70,2.462476,83.656032
England,7,98,2.286187,81.268320
Belgium,7,95,1.830037,78.183429
Colombia,4,57,1.858291,75.135303
Argentina,4,56,1.398924,74.341428


In [9]:
player_match_dataset[
    [
        "match_id",
        "player.name",
        "team.name",
        "opponent",
        "result",
        "minutes_played",
        "passes_per90",
        "shots_per90",
        "total_xg_per90",
        "team_momentum",
        "team_xg",
    ]
].sort_values("total_xg_per90", ascending=False).head(25)


,match_id,player.name,team.name,opponent,result,minutes_played,passes_per90,shots_per90,total_xg_per90,team_momentum,team_xg
1292,7552,Michy Batshuayi Tunga,Belgium,Tunisia,win,26,20.769231,20.769231,7.686508,131.444057,5.336422
1122,7583,Roberto Firmino Barbosa de Oliveira,Brazil,Mexico,win,11,49.090909,8.181818,7.544009,79.535813,3.320421
309,7585,Marcus Rashford,England,Colombia,draw,10,36.000000,9.000000,7.051500,133.465739,6.161790
176,7534,Mario Gómez García,Germany,Mexico,loss,12,15.000000,22.500000,6.662882,80.545553,2.299259
916,7581,Milan Badelj,Croatia,Denmark,draw,13,34.615385,6.923077,5.424231,135.442435,6.087736
545,7557,Karim Ansarifard,Iran,Portugal,draw,18,10.000000,5.000000,3.917500,44.210818,1.576970
1684,8652,Alan Dzagoev,Russia,Croatia,draw,21,47.142857,4.285714,3.357857,124.753704,4.988439
919,7581,Michael Krohn-Dehli,Denmark,Croatia,draw,22,45.000000,4.090909,3.205227,108.225267,4.613995
1186,7535,Filip Kostić,Serbia,Costa Rica,win,29,18.620690,6.206897,2.866216,62.448536,1.992569
900,7581,Andrej Kramarić,Croatia,Denmark,draw,26,51.923077,6.923077,2.832054,135.442435,6.087736


## Zapis do CSV

Plik trafia do `outputs/tables/player_match_dataset.csv`. Katalog `outputs/` jest miejscem na wygenerowane artefakty.


In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
player_match_dataset.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Zapisano:", OUTPUT_PATH)
print("Liczba wierszy:", len(player_match_dataset))


Zapisano: c:\Users\misie\Documents\UAM\MGR_MK\outputs\tables\player_match_dataset.csv
Liczba wierszy: 1787
